# ETL — IPCA (IBGE Tabela 1737)
**TCC — Business Intelligence aplicado ao BPS**  
Mônica Anatália — POLI USP PRO

**O que este notebook faz:**
1. Lê o CSV da Tabela 1737 do SIDRA/IBGE
2. Transpõe de formato wide para long (uma linha por mês)
3. Calcula o fator de deflação para cada mês (base: dezembro/2023)
4. Salva `ipca_deflator.parquet` — tabela de referência para o Power BI

**Como usar o deflator no Power BI:**  
`preco_real = preco_unitario × fator_deflacao`  
Isso converte qualquer preço histórico para valores de dezembro/2023, último mês da série analisada.

## 1. Montar o Google Drive

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────


## 2. Instalar dependências

In [2]:
!pip install pyarrow --quiet
print('Dependências instaladas ✓')

## 3. Configuração
> ⚙️ Ajuste os caminhos abaixo.

> `MES_REFERENCIA` define a base do deflator — use sempre o último mês  
> completo disponível na sua série analisada (dezembro de 2023).

In [5]:
import pandas as pd
from pathlib import Path

# ── AJUSTE AQUI ──────────────────────────────────────────────────────────
ARQUIVO_IPCA    = Path(PASTA_DADOS) / 'IPCA Bruto/tabela1737.csv'
PASTA_SAIDA     = Path(PASTA_DADOS) / 'IPCA Tratado'
MES_REFERENCIA  = 'dezembro 2023'   # base do deflator = ultimo mes da serie analisada
ANO_INICIO      = 2000              # cobre toda a base harmonizada; a analise usa 2009-2023
ANO_FIM         = 2023              # ultimo ano da serie analisada (recorte 2009-2023)
# ─────────────────────────────────────────────────────────────────────────

Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)
print(f'Arquivo IPCA : {ARQUIVO_IPCA}')
print(f'Pasta saída  : {PASTA_SAIDA}')
print(f'Mês base     : {MES_REFERENCIA}')
print(f'Período BPS  : {ANO_INICIO}–{ANO_FIM}')

## 4. Ler e Processar o CSV do IBGE
> O SIDRA exporta os meses nas colunas (formato wide).  
> Aqui transpomos para formato long: uma linha por mês.

In [6]:
# Dicionário de meses em português
MESES_PT = {
    'janeiro':'01','fevereiro':'02','março':'03','abril':'04',
    'maio':'05','junho':'06','julho':'07','agosto':'08',
    'setembro':'09','outubro':'10','novembro':'11','dezembro':'12'
}

def converter_mes(s):
    partes = str(s).strip().split(' ')
    if len(partes) == 2:
        return f"{partes[1]}-{MESES_PT.get(partes[0], '00')}-01"
    return None

# Lê cabeçalho (linha 3) — contém os nomes dos meses
df_header = pd.read_csv(
    ARQUIVO_IPCA, sep=';', encoding='utf-8-sig',
    skiprows=3, nrows=1, header=None
)
meses_raw = df_header.iloc[0, 1:].values

# Lê linha de dados (linha 4) — contém os índices
df_dados = pd.read_csv(
    ARQUIVO_IPCA, sep=';', encoding='utf-8-sig',
    skiprows=4, nrows=1, header=None
)
valores_raw = df_dados.iloc[0, 1:].values

# Monta DataFrame long
df_ipca = pd.DataFrame({'mes_ano': meses_raw, 'indice_ipca': valores_raw})

# Limpa vírgula decimal
df_ipca['indice_ipca'] = (
    df_ipca['indice_ipca'].astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# Converte mes_ano para datetime
df_ipca['data']  = pd.to_datetime(df_ipca['mes_ano'].apply(converter_mes))
df_ipca['ano']   = df_ipca['data'].dt.year
df_ipca['mes']   = df_ipca['data'].dt.month

print(f'Total de meses no arquivo: {len(df_ipca)}')
print(f'Período: {df_ipca["mes_ano"].iloc[0]} → {df_ipca["mes_ano"].iloc[-1]}')
print(df_ipca[['mes_ano','indice_ipca','ano','mes']].tail(3).to_string(index=False))

## 5. Calcular Fator de Deflação
> `fator_deflacao = índice_referência / índice_do_mês`  
> Multiplique qualquer preço histórico por este fator para obter o valor em `MES_REFERENCIA`.

In [7]:
# Índice do mês de referência
mask_ref = df_ipca['mes_ano'].str.strip() == MES_REFERENCIA
if not mask_ref.any():
    raise ValueError(f'Mês de referência "{MES_REFERENCIA}" não encontrado no arquivo.')

idx_ref = df_ipca.loc[mask_ref, 'indice_ipca'].values[0]
print(f'Índice de referência ({MES_REFERENCIA}): {idx_ref:.2f}')

# Calcula fator
df_ipca['fator_deflacao'] = idx_ref / df_ipca['indice_ipca']

# Filtra apenas o período do BPS
df_bps = df_ipca[df_ipca['ano'].between(ANO_INICIO, ANO_FIM)].copy()
df_bps = df_bps[['ano','mes','data','mes_ano','indice_ipca','fator_deflacao']].reset_index(drop=True)

print(f'Meses no período BPS ({ANO_INICIO}–{ANO_FIM}): {len(df_bps)}')
print(f'\nFator jan/2000 : {df_bps["fator_deflacao"].iloc[0]:.4f}x')
print(f'Fator dez/2025 : {df_bps["fator_deflacao"].iloc[-1]:.4f}x (deve ser 1.0000)')
print(f'\nAmostra:')
print(df_bps[['mes_ano','indice_ipca','fator_deflacao']].iloc[[0,1,2,-3,-2,-1]].to_string(index=False))

## 6. Exportar
> Salva `ipca_deflator.parquet` na pasta `Base Tratamento Minimo`.

In [8]:
saida = Path(PASTA_SAIDA) / 'ipca_deflator.parquet'
df_bps.to_parquet(saida, index=False)

# Valida lendo de volta
df_val = pd.read_parquet(saida)
print(f'Arquivo salvo: {saida}')
print(f'Linhas : {len(df_val)}')
print(f'Colunas: {list(df_val.columns)}')
print(f'\nPrévia:')
print(df_val.head(3).to_string(index=False))

## 7. Como usar no Power BI

### Relacionamento
No modelo do Power BI, crie um relacionamento entre:
- `ipca_deflator[ano]` + `ipca_deflator[mes]` → `BPS_AAAA[ano]` + mês extraído de `data_compra`

### Medida DAX — Preço Real
```dax
Preco Real (dez/2023) =
SUMX(
    fatos_bps,
    fatos_bps[preco_unitario] * RELATED(ipca_deflator[fator_deflacao])
)
```

### Interpretação
- `fator_deflacao` de janeiro/2000 ≈ 4,24 → preços de 2000 valem 4,24× mais em termos de 2023
- Use `Preco Real` nos visuais longitudinais para comparar décadas sem distorção inflacionária